# Vegetation classification with a random forest: repeated random 70/30 splits

This notebook is an **example workflow**. It trains a random forest classifier on segment-level predictor variables and evaluates it against labelled reference segments. The labelled segments are pooled and divided with a **stratified random 70/30 split** (70% of each class for training). The single-run section uses the split drawn with `split_seed`; the averaged-results section draws a new split for every run (split seed and forest seed = run number) and reports averaged confusion matrices and the mean ± SD of class-level accuracy.

**Data:** the data used in our study contain proprietary information and are not distributed with this repository. The notebook is intended to be run on your own data. Supply a geopackage (or other GIS file that GeoPandas can read) containing a segment layer with:

* a **class label** column (`Class` in our data), with integer class codes;
* a column marking the **labelled segments** (`training_type` in our data, with the value `training` or `truth` for labelled segments; segments with any other value or none are unlabelled and are only classified). Because the notebook draws its own random split, it does not matter which labelled segments are marked `training` and which `truth`;
* a **segment ID** column (`SegmentID`) and, optionally, a **plot ID** column (`PlotID`), which are written to the output tables;
* one column per **predictor variable**.

You can use the same field names as we did or your own. If your fields differ, change the names in the *Paths*, *Load the data* and *Predictor variables* cells (and the `SegmentID` / `PlotID` names in the output cells near the end). Our predictor list is kept in the *Predictor variables* cell as an example of the variables we used; replace it with the columns in your data.

**Class codes:** the grouped confusion matrices (levels 3, 2 and 1) group classes by the first 3, 2 and 1 digits of the class code, which matches our hierarchical classification scheme. If your codes are not hierarchical in this way, skip or adapt those cells.

**Outputs:** each run writes to a new time-stamped folder in `../rf_outputs/`: the trained model, run metadata, variable-importance charts, confusion matrices (class level and grouped levels) with user's and producer's accuracy, class-level importance, per-segment predictions, and averaged results over repeated runs.

**Settings:** `model_run_name` (used in output names) and `class_weight` (`None`, `"balanced"` or `"balanced_subsample"`) are set in the *Run name and settings* cell; the number of repeated runs (`n_runs`) is set at the start of the averaged-results section.

**Requirements:** Python 3.10+, `geopandas` (with `pyogrio` or `fiona`), `pandas`, `numpy`, `scikit-learn`, `matplotlib`, `seaborn`, `joblib`. Reading a large file can take several minutes.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelBinarizer
import seaborn as sn
import numpy as np
import math
import os
from datetime import datetime
import json
from sklearn.metrics import  confusion_matrix
import time
from joblib import dump, load
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import geopandas as gpd


## Paths

Set `geopackage` to your own data file. The example file name below is the one used in our study; the file itself is not provided.

In [ ]:
# Paths are relative to the folder this notebook lives in (Jupyter runs notebooks from their own folder).
# Input:  your geopackage (or other GeoPandas-readable file); edit the file name below.
#         The name shown is our study's file, which is not distributed.
# Output: each run gets a time-stamped folder inside ../rf_outputs/
script_dir = os.path.abspath('')
geopackage = os.path.join(script_dir, 'pilot_project_RFv12.1.gpkg')
basedir = os.path.join(os.path.dirname(script_dir), 'rf_outputs')
if not os.path.exists(geopackage):
    raise FileNotFoundError(f"Geopackage not found: {geopackage}")


## Load the data

The segment layer is found by name: edit `'MergedSegments'` in the next cell to match part of your layer name (or set `classification_level_layer` directly).

In [ ]:
# List layers without needing GDAL's Python bindings
try:
    layer_names = [name for name, _ in gpd.list_layers(geopackage).values]
except AttributeError:  # geopandas < 1.0
    import fiona
    layer_names = fiona.listlayers(geopackage)

classification_level_layer=None
for layer_name in layer_names:
    if 'MergedSegments'.lower() in layer_name.lower():
        classification_level_layer=layer_name
if classification_level_layer is None:
    raise Exception("Could not find classification level layer in geopackage")
print(classification_level_layer)


In [ ]:

#Read in data to a data frame skipping the geom.
#This will be the basis of the output dataframe.
segment_df = gpd.read_file(geopackage, layer=classification_level_layer,ignore_geometry=True)


**Random stratified 70/30 split**

The labelled segments (those with a value in the `training_type` column) are pooled, and for each class 70% are sampled for training and the rest are kept for truth.
`split_seed` fixes the draw so the run can be reproduced; the averaged section further down draws a new split for every run.
Any split already stored in your data is kept in `original_training_type` for reference. In our data the column held `training` / `truth`; if your labelled segments are marked differently, edit `labelled_mask` in the next cell.

In [ ]:
train_fraction = 0.7   # fraction of each class used for training
split_seed = 0         # seed for the split used in the single-run section

def random_split(labelled_df, train_fraction, seed):
    """Stratified random split: per class, sample train_fraction of the segments for training.
    Returns a Series of 'training' / 'truth' aligned to labelled_df."""
    split = pd.Series('truth', index=labelled_df.index)
    for class_id in sorted(labelled_df['Class'].unique()):
        this_class_rows = labelled_df[labelled_df['Class'] == class_id]
        training_rows = this_class_rows.sample(frac=train_fraction, random_state=seed)
        split.loc[training_rows.index] = 'training'
    return split

segment_df['original_training_type'] = segment_df['training_type']
labelled_mask = segment_df['training_type'].isin(['training', 'truth'])
segment_df.loc[labelled_mask, 'training_type'] = random_split(segment_df.loc[labelled_mask], train_fraction, split_seed)
segment_df.loc[labelled_mask].groupby(['Class', 'training_type']).size().unstack(fill_value=0)


In [ ]:
training_df=segment_df.loc[segment_df['training_type'] == 'training']
training_df

In [ ]:
# get the class counts of the training data
training_df.groupby(['Class']).size()

In [ ]:
truthy_df=segment_df.loc[segment_df['training_type'] == 'truth']
truthy_df

In [ ]:
#get the count of each class
truthy_df.groupby(['Class']).size()

In [ ]:
unique_class_list=training_df.Class.unique()
unique_class_list



## Run name and settings

In [ ]:
model_run_name='RFv12 random split'

# Class weighting for the random forest. Options:
#   None                  -> no weighting (original behaviour)
#   "balanced"            -> weights inversely proportional to class frequency in the full training set
#   "balanced_subsample"  -> same, but recomputed on each tree's bootstrap sample
class_weight = None

# The weighting is added to the run name so output folders, file names and chart titles show which one was used
if class_weight is not None:
    model_run_name = model_run_name + ' cw-' + class_weight
print(model_run_name)


In [ ]:
# Define a prefix for the output file name
file_prefix='-'+model_run_name.replace(' ','_')

# basedir (../rf_outputs) is set in the paths cell at the top

# Get the current date and time
now = datetime.now()

# Format the date and time as a string
date_str = now.strftime("%m-%d-%Y_%H_%M_%S")

# Add the file prefix to the date and time string
date_str = date_str + file_prefix
# date_str='04-27-2023_11_13_54-PCA-RFv3'
# Combine the base directory and the output file name to get the full path of the output directory
output_directory=os.path.join(basedir, date_str)

# Print the output directory path
print(output_directory)

# Create the output directory if it doesn't already exist
if not os.path.exists(output_directory):
   os.makedirs(output_directory, exist_ok=True)
else:
   print('Output directory already exists.')



List the available columns; the predictor set is chosen in the next section.

In [ ]:
full_test_train_df=segment_df.loc[segment_df['training_type'].isin(['training','truth'])]
full_test_train_df

In [ ]:
all_feature_columns=list(training_df.columns)
all_feature_columns

## Predictor variables

The list below holds the predictor fields from our study and is included as an example. Replace it with the predictor columns in your own data, and keep the class label column (`Class`) at the end of the list.

In [ ]:
# Example: the predictor fields used in our study. Replace with your own column names;
# the last entry is the class label column.
custom_features=['GLCM_Homog', 'GLCM_Mean_', 'Mean_Aspec', 'Mean_Blue', 'Mean_DEM', 'Mean_gedi_', 'Mean_Green', 'Mean_NDMI', 'Mean_NIR', 'Mean_Red', 'Mean_SolRa', 'Mean_TWI', 'Median_DHM', 'Mn_smap_10', 'Mn_smap_c1', 'Mn_smap_c2', 'Mn_smap_c3', 'Mn_smap_c4', 'Mn_smap_c5', 'Mn_smap_c6', 'Mn_smap_c7', 'Mn_smap_c8', 'Mn_smap_c9', 'R_Grass', 'R_high_tree', 'R_low_shrub', 'R_low_tree', 'R_med_shrub', 'Rel_Bare', 'St_smap_10', 'Std_gedi', 'Std_NDMI', 'St_smap_c9', 'St_smap_c8', 'St_smap_c7', 'St_smap_c6', 'St_smap_c2', 'St_smap_c5', 'St_smap_c4', 'St_smap_c3', 'St_smap_c1', 'Std_Blue', 'Std_DEM', 'Std_DHM', 'Std_Green', 'Std_NIR', 'Std_Red', 'Std_SolRad', 'Std_TWI', 'Mean_PCA1', 'Mean_PCA2', 'Mean_PCA3', 'Mean_PCA4', 'Mean_PCA5', 'Mean_PCA6', 'Mean_PCA7', 'Mean_PCA8', 'Std_PCA1', 'Std_PCA2', 'Std_PCA3', 'Std_PCA4', 'Std_PCA5', 'Std_PCA6', 'Std_PCA7', 'Std_PCA8' ,'Class']


Select the predictor columns (or reuse the column list from an existing `metadata.json`).

In [ ]:
if os.path.exists(os.path.join(output_directory,'metadata.json')):
    with open(os.path.join(output_directory,'metadata.json')) as f:
        data = json.load(f)
        print('Using metadata.json for columns list')
        cols=data['columns']
else:
    cols=custom_features
print(cols)
training_usable_df = training_df[cols]
truthy_usable_df = truthy_df[cols]
full_test_train_usable_df = full_test_train_df[cols]



**Split the variables and the classes**

In [ ]:


# Get the training variables by dropping the "Class" column from the training data frame
variables_train = training_usable_df.drop('Class', axis='columns')

# Get the class names for the training data
class_names_train = training_usable_df.Class

# Get the variables for the truth data by dropping the "Class" column from the truth data frame
variables_truth = truthy_usable_df.drop('Class', axis='columns')

# Get the class names for the truth data
class_names_truth = truthy_usable_df.Class

# Determine the maximum number of features to consider when splitting a node in a decision tree based
# on the number of variables in the training data frame
max_features = len(variables_train.columns)



## Train the model

In [ ]:
scores=[]
score=0

One model is trained per run of this section.

In [ ]:
# Train a single random forest on the training data and score it on the truth set
model = RandomForestClassifier(class_weight=class_weight)
model.fit(variables_train, class_names_train)
score = model.score(variables_truth, class_names_truth)
scores.append(score)
print(score)


In [ ]:
print("Score:"+str(score))


***Save the model.***

In [ ]:
dump(model, os.path.join(output_directory,'model.joblib') )

Calculate impurity-based (Gini) variable importances.

In [ ]:
# Create a copy of 'cols' list and remove the 'Class' label from it.
feature_names = cols.copy()
feature_names.remove('Class')

# Calculate feature importances using the trained random forest classifier model.
importances = model.feature_importances_

# Compute the standard deviation of feature importances across all trees in the forest.
std = np.std([tree.feature_importances_ for tree in model.estimators_], axis=0)

# Create a Pandas Series object with feature importances and the corresponding feature names as indices.
forest_importances = pd.Series(importances, index=feature_names)

# Create a dictionary of feature importances indexed by feature names.
gini_importances = {}
for feature_name, importance_score in zip(feature_names, importances):
    gini_importances[feature_name] = importance_score

Save run metadata.

In [ ]:
# Create an empty dictionary to store metadata information.
json_obj = {}

# Add the model score, number of training rows, number of truth rows, columns used,
# and gini importances to the dictionary.
json_obj['score'] = model.score(variables_truth, class_names_truth)
json_obj['training_rows'] = len(training_df)
json_obj['truth_rows'] = len(truthy_df)
json_obj['columns'] = cols
json_obj['gini'] = gini_importances
json_obj['run_name']=model_run_name
json_obj['class_weight']=class_weight
json_obj['split']=f'stratified random {train_fraction:.0%}/{1-train_fraction:.0%}, split_seed={split_seed}'
# Convert the dictionary to a JSON object with indentation for readability.
json_object = json.dumps(json_obj, indent=4)

# Write the JSON object to a file in the output directory.
json_file = os.path.join(output_directory, 'metadata.json')
with open(json_file, "w") as outfile:
    outfile.write(json_object)



***We will now classify our test set using the trained model.***


***The output is predicted class for each of the test rows***

In [ ]:
y_predicted = model.predict(variables_truth)


To make this a little more understandable lets create a confusion matrix

In [ ]:
cm = confusion_matrix(class_names_truth, y_predicted,normalize='true')
cm_count = confusion_matrix(class_names_truth, y_predicted)
print(str(cm_count))

***Let's make that a little more easy to understand***

***First we need convert the sklearn confusion matrix to a pandas data frame.***



***This will allow us to have more control over the tick labels***



***This function will do the conversion***

In [ ]:
def cm2df(cm, ylabels, xlabels):
    # rows = reference/truth classes (xlabels), columns = predicted classes (ylabels),
    # matching sklearn's confusion_matrix layout
    return pd.DataFrame(cm, index=list(xlabels), columns=list(ylabels))


Get a unique ordered list of the classes for use in charts later on.

In [ ]:
unique=[]
class_names_test_list=[]
for cls in class_names_truth:
    if cls not in unique:
        unique.append(int(cls))
    class_names_test_list.append(int(cls))
for cls in y_predicted:
    if cls not in unique:
        unique.append(cls)
unique.sort()


In [ ]:
class_names_test_list = class_names_truth.unique().tolist()
class_names_test_list.sort()
class_counts_training = full_test_train_usable_df["Class"].value_counts()
class_counts_training_sorted = class_counts_training.reindex(class_names_test_list)
print(class_counts_training_sorted.to_json())

In [ ]:
cm_df_unique = cm2df(cm, unique,unique)
cm_df_unique.to_csv(os.path.join(output_directory,'cm-'+date_str+".csv"))

## Charts

Variable importance charts.

In [ ]:


# Plot feature importances using a bar chart and save it as an image in the output directory.
# Create a new figure and axis pair.
fig, ax = plt.subplots()

# Set the height and width of the figure in inches.
fig.set_figheight(10)
fig.set_figwidth(20)
ax.set_facecolor('white')
# Plot bar chart of feature importances with error bars as given by std.
forest_importances.plot.bar(yerr=std, ax=ax)

# Set the title and y-axis label of the plot.
ax.set_title("Feature importances using MDI\n("+model_run_name+")")
ax.set_ylabel("Mean decrease in impurity")

# Save the plot as a PNG image file.
fig.savefig(os.path.join(output_directory, "importances-using MDI-" + date_str + ".png"))

# Adjust the spacing between subplots to minimize overlapping text.
fig.tight_layout()


In [ ]:
forest_importances_sorted=forest_importances.sort_values(ascending=False)
# Plot feature importances using a bar chart and save it as an image in the output directory.
# Create a new figure and axis pair.
fig, ax = plt.subplots()

# Set the height and width of the figure in inches.
fig.set_figheight(10)
fig.set_figwidth(20)
ax.set_facecolor('white')
#reorder the std


# Plot bar chart of feature importances with error bars as given by std.
forest_importances_sorted.plot.bar( ax=ax)

# Set the title and y-axis label of the plot.
ax.set_title("Feature Importances Using MDI Sorted Descending\n("+model_run_name+")")
ax.set_ylabel("Mean decrease in impurity")

# Save the plot as a PNG image file.
fig.savefig(os.path.join(output_directory, "importances-using MDI-sorted-" + date_str + ".png"))
#fig.savefig(os.path.join(output_directory, "300-dpi-importances-using MDI-sorted-" + date_str + ".png"),dpi=300)

# Adjust the spacing between subplots to minimize overlapping text.
fig.tight_layout()

Normalized confusion matrix (finest class level).

In [ ]:

color_obj={}
cm_df = cm2df(cm, unique,unique)
px = 1/plt.rcParams['figure.dpi']
plt.rcParams.update({'font.size': 35})
plt.figure(figsize=(5500*px, 3500*px))
cm_df_clean = cm_df.replace(0, np.nan)
# Calculate user and producer accuracy values
user_accuracy = np.sum(cm, axis=0)
producer_accuracy =  np.sum(cm, axis=1)

# Replace any NaN values with 0 in the accuracy calculations
user_accuracy = [0 if math.isnan(x) else x for x in user_accuracy]
producer_accuracy = [0 if math.isnan(x) else x for x in producer_accuracy]

# Add producer's accuracy column to cleaned confusion matrix
cm_df_clean["truth"] = producer_accuracy

# Append NaN value to bottom row of accuracy matrix
user_accuracy = np.append(user_accuracy, np.nan)
cm_df_clean.loc["Predicted Total"] = user_accuracy
ax=sn.heatmap(cm_df_clean, annot=True,cmap='Greens')

labels = unique + ["Truth Total"]
ax.set_xticklabels(labels, fontsize=20)
ax.set_facecolor('white')

for i, label in enumerate(ax.get_xticklabels()):
    for key in color_obj.keys():
        if label.get_text().startswith(key):
            label.set_color(color_obj[key]['text'])
            label.set_bbox(dict(facecolor=color_obj[key]['backgroud'], edgecolor='none', pad=9))
for i, label in enumerate(ax.get_yticklabels()):
    for key in color_obj.keys():
        if label.get_text().startswith(key):
            label.set_color(color_obj[key]['text'])
            label.set_bbox(dict(facecolor=color_obj[key]['backgroud'], edgecolor='none', pad=6))

ax.set_facecolor('white')
plt.setp(ax.get_xticklabels()[:-1], rotation=0)
plt.setp(ax.get_xticklabels()[-1:], rotation=90)

sn.set(font_scale=1) 
plt.title('Vegetation Classification Confusion Matrix (Normalized)\n('+model_run_name+')', pad=40, fontsize=35)
plt.xlabel('Predicted', fontsize=55)
plt.ylabel('Truth', fontsize=55)
plt.yticks(rotation=0, fontsize=30)
plt.xticks(fontsize=30)
plt.savefig(os.path.join(output_directory,"cm-"+date_str+".png"))
plt.show()


Count confusion matrix with user's and producer's accuracy.

In [ ]:
# Convert cm_count to a pandas dataframe using unique values for labelling
cm_df = cm2df(cm_count, unique,unique)

# Calculate the pixel ratio and set figure size accordingly
px = 1/plt.rcParams['figure.dpi']
plt.figure(figsize=(5500*px, 2500*px))

# Update font size of plot parameters
plt.rcParams.update({'font.size': 35})

# Replace 0s in the matrix with NaN values
cm_df_clean = cm_df.replace(0, np.nan)

# Calculate user and producer accuracy values
# Rows = reference (truth), columns = classified (predicted)
# User's accuracy     = correct / column total (per classified class)  -> bottom row
# Producer's accuracy = correct / row total    (per reference class)   -> right-hand column
user_accuracy = np.diag(cm_count) / np.sum(cm_count, axis=0)
producer_accuracy = np.diag(cm_count) / np.sum(cm_count, axis=1)

# Replace any NaN values with 0 in the accuracy calculations
user_accuracy = [0 if math.isnan(x) else x for x in user_accuracy]
producer_accuracy = [0 if math.isnan(x) else x for x in producer_accuracy]

# Add producer's accuracy column to cleaned confusion matrix
cm_df_clean["Producer's Accuracy"] = producer_accuracy

# Append NaN value to bottom row of accuracy matrix
user_accuracy = np.append(user_accuracy, np.nan)
cm_df_clean.loc["User's Accuracy"] = user_accuracy

# Set seaborn font scale parameter
sn.set(font_scale=3)
class_counts = [int(class_counts_training_sorted[k]) for k in unique]
# Create heatmap with formatted annotations
plt.figure(figsize=(5500*px, 2500*px))
annot_format = [["{:.0f}".format(val) if (row != len(cm_df_clean) - 1) and (col != len(cm_df_clean.columns) - 1) else "{:.2f}".format(val) for col, val in enumerate(row_vals)] for row, row_vals in enumerate(cm_df_clean.values)]
ax = sn.heatmap(cm_df_clean, annot=annot_format, fmt="", cmap='Greens', linewidths=.5)


labels = [f"{class_name}\n({class_count})" for class_name, class_count in zip(unique, class_counts)]
labels.append("Producer's Accuracy")
ax.set_xticklabels(labels, fontsize=20)
plt.setp(ax.get_xticklabels()[:-1], rotation=0)
plt.setp(ax.get_xticklabels()[-1:], rotation=90)

# Set background color of heatmap to white
ax.set_facecolor('white')


for i, label in enumerate(ax.get_xticklabels()):
    for key in color_obj.keys():
        if label.get_text().startswith(key):
            label.set_color(color_obj[key]['text'])
            label.set_bbox(dict(facecolor=color_obj[key]['backgroud'], edgecolor='none', pad=11))
for i, label in enumerate(ax.get_yticklabels()):
    for key in color_obj.keys():
        if label.get_text().startswith(key):
            label.set_color(color_obj[key]['text'])
            label.set_bbox(dict(facecolor=color_obj[key]['backgroud'], edgecolor='none', pad=13))

# Set plot title, x and y labels, and tick parameters
plt.title('Vegetation Classification Confusion Matrix\n('+model_run_name+')', fontsize=35, pad=40)
plt.xlabel('Classified', fontsize=50)
plt.ylabel('Reference', fontsize=50)
plt.yticks(rotation=0, fontsize=30)
plt.xticks(fontsize=30)
ax.text(-0.1, -0.15, "Class Counts are shown in parentheses", fontsize=50, transform=ax.transAxes, va="top", ha="left")
ax.text(0.98, 0.0048, str(round(score,3)), transform=ax.transAxes)
# Save and show the plot
plt.savefig(os.path.join(output_directory, "cm_count-" + date_str + ".png"))
#plt.savefig(os.path.join(output_directory, "300-dpi-cm_count-" + date_str + ".png"),dpi=300)
plt.show()



Group values at level 3 and make a confusion matrix with the grouped data.

In [ ]:
color_obj_level_3={}
depth=2
for class_id  in unique:
    #if the first two digets are not in color_obj, add them as a key and assign green or light green alternating between them.
    if str(class_id)[:depth] not in color_obj_level_3:
        if len(color_obj_level_3)%2==0:
            color_obj_level_3[str(class_id)[:depth]]={'backgroud':'#E5E4E2','text':'black'}
        else:
            color_obj_level_3[str(class_id)[:depth]]={'backgroud':'#faf7f7','text':'black'}



cm_count_df_unique = cm2df(cm_count, unique,unique)
cm_count_df_unique.to_csv(os.path.join(output_directory,'cm_count'+date_str+".csv"))
grouped_x = cm_count_df_unique.T.groupby(lambda x: str(x)[:3]).sum().T
grouped_y = grouped_x.groupby(lambda x: str(x)[:3], axis=0).sum()
grouped_y.to_csv(os.path.join(output_directory,'cm_count_group_level_3-'+date_str+".csv"))
# Calculate user's accuracy
user_acc = np.diag(grouped_y) / grouped_y.sum(axis=0)
user_acc = user_acc.fillna(0)
# Calculate producer's accuracy
prod_acc = np.diag(grouped_y) / grouped_y.sum(axis=1)
prod_acc = prod_acc.fillna(0)
# Round the accuracies to 2 decimal places
print(user_acc)
print(prod_acc)
user_acc = np.round(user_acc, 2)
prod_acc = np.round(prod_acc, 2)


px = 1/plt.rcParams['figure.dpi']

plt.rcParams.update({'font.size': 45})
plt.figure(figsize=(5500*px, 2800*px))
grouped_y_clean = grouped_y.replace(0, np.nan)

plt.title('Vegetation Classification Confusion Matrix Level 3\n('+model_run_name+')', fontsize=35, pad=40)
# Add producer's accuracy column to cleaned confusion matrix
grouped_y_clean["Producer's Accuracy"] = prod_acc

# Append NaN value to bottom row of accuracy matrix
user_acc = np.append(user_acc, np.nan)
grouped_y_clean.loc["User's Accuracy"] = user_acc


ax=sn.heatmap(grouped_y_clean, annot=True,cmap='Greens')
ax.set_facecolor('white')
accuracy = np.diag(grouped_y).sum() / grouped_y.to_numpy().sum()
ax.text(0.97, 0.01, str(round(accuracy,2)), transform=ax.transAxes)

sn.set(font_scale=1) 
#plt.title('Vegetation Classification Confusion Matrix Level 3\nUser Accuracy: {}\nProducer Accuracy: {}'.format(user_acc, prod_acc), fontsize=75)
plt.xlabel('Predicted', fontsize=45)
plt.ylabel('Truth', fontsize=45)
plt.yticks(rotation=0, fontsize=45)
plt.xticks(fontsize=45)

pad_val=15


# Apply the customized styling to the class labels
for i, label in enumerate(ax.get_xticklabels()):
    for key in color_obj_level_3.keys():
        if label.get_text().startswith(key):
            label.set_color(color_obj_level_3[key]['text'])
            label.set_bbox(dict(facecolor=color_obj_level_3[key]['backgroud'], edgecolor='none', pad=pad_val))
for i, label in enumerate(ax.get_yticklabels()):
    for key in color_obj_level_3.keys():
        if label.get_text().startswith(key):
            label.set_color(color_obj_level_3[key]['text'])
            label.set_bbox(dict(facecolor=color_obj_level_3[key]['backgroud'], edgecolor='none', pad=pad_val))
    
plt.savefig(os.path.join(output_directory,"cm_count_group_level_3-"+date_str+".png"))

plt.show()

In [ ]:
color_obj_level={}
depth=1
for class_id  in unique:
    #if the first two digets are not in color_obj, add them as a key and assign green or light green alternating between them.
    if str(class_id)[:depth] not in color_obj_level:
        if len(color_obj_level)%2==0:
            color_obj_level[str(class_id)[:depth]]={'backgroud':'#E5E4E2','text':'black'}
        else:
            color_obj_level[str(class_id)[:depth]]={'backgroud':'#faf7f7','text':'black'}



cm_count_df_unique = cm2df(cm_count, unique,unique)
cm_count_df_unique.to_csv(os.path.join(output_directory,'cm_count'+date_str+".csv"))
grouped_x = cm_count_df_unique.T.groupby(lambda x: str(x)[:2]).sum().T
grouped_y = grouped_x.groupby(lambda x: str(x)[:2], axis=0).sum()
grouped_y.to_csv(os.path.join(output_directory,'cm_count_group_level_2-'+date_str+".csv"))
# Calculate user's accuracy
user_acc = np.diag(grouped_y) / grouped_y.sum(axis=0)
user_acc = user_acc.fillna(0)
# Calculate producer's accuracy
prod_acc = np.diag(grouped_y) / grouped_y.sum(axis=1)
prod_acc = prod_acc.fillna(0)
# Round the accuracies to 2 decimal places
user_acc = np.round(user_acc, 2)
prod_acc = np.round(prod_acc, 2)


px = 1/plt.rcParams['figure.dpi']
plt.rcParams.update({'font.size': 45})
plt.figure(figsize=(5500*px, 2800*px))
grouped_y_clean = grouped_y.replace(0, np.nan)
plt.title('Vegetation Classification Confusion Matrix Level 2\n('+model_run_name+')', fontsize=35, pad=40)

# Add producer's accuracy column to cleaned confusion matrix
grouped_y_clean["Producer's Accuracy"] = prod_acc

# Append NaN value to bottom row of accuracy matrix
user_acc = np.append(user_acc, np.nan)
grouped_y_clean.loc["User's Accuracy"] = user_acc


ax=sn.heatmap(grouped_y_clean, annot=True,cmap='Greens')
ax.set_facecolor('white')
accuracy = np.diag(grouped_y).sum() / grouped_y.to_numpy().sum()
ax.text(0.97, 0.01, str(round(accuracy,2)), transform=ax.transAxes)

sn.set(font_scale=1) 
#plt.title('Vegetation Classification Confusion Matrix Level 3\nUser Accuracy: {}\nProducer Accuracy: {}'.format(user_acc, prod_acc), fontsize=75)
plt.xlabel('Predicted', fontsize=45)
plt.ylabel('Truth', fontsize=45)
plt.yticks(rotation=0, fontsize=45)
plt.xticks(fontsize=45)

pad_val=15


# Apply the customized styling to the class labels
for i, label in enumerate(ax.get_xticklabels()):
    for key in color_obj_level.keys():
        if label.get_text().startswith(key):
            label.set_color(color_obj_level[key]['text'])
            label.set_bbox(dict(facecolor=color_obj_level[key]['backgroud'], edgecolor='none', pad=pad_val))
for i, label in enumerate(ax.get_yticklabels()):
    for key in color_obj_level.keys():
        if label.get_text().startswith(key):
            label.set_color(color_obj_level[key]['text'])
            label.set_bbox(dict(facecolor=color_obj_level[key]['backgroud'], edgecolor='none', pad=pad_val))
    
plt.savefig(os.path.join(output_directory,"cm_count_group_level_2-"+date_str+".png"))

plt.show()

In [ ]:
color_obj_level={}
depth=0
for class_id  in unique:
    #if the first two digets are not in color_obj, add them as a key and assign green or light green alternating between them.
    if str(class_id)[:depth] not in color_obj_level:
        if len(color_obj_level)%2==0:
            color_obj_level[str(class_id)[:depth]]={'backgroud':'#E5E4E2','text':'black'}
        else:
            color_obj_level[str(class_id)[:depth]]={'backgroud':'#faf7f7','text':'black'}



cm_count_df_unique = cm2df(cm_count, unique,unique)
cm_count_df_unique.to_csv(os.path.join(output_directory,'cm_count'+date_str+".csv"))
grouped_x = cm_count_df_unique.T.groupby(lambda x: str(x)[:1]).sum().T
grouped_y = grouped_x.groupby(lambda x: str(x)[:1], axis=0).sum()
grouped_y.to_csv(os.path.join(output_directory,'cm_count_group_level_1-'+date_str+".csv"))
# Calculate user's accuracy
user_acc = np.diag(grouped_y) / grouped_y.sum(axis=0)
user_acc = user_acc.fillna(0)
# Calculate producer's accuracy
prod_acc = np.diag(grouped_y) / grouped_y.sum(axis=1)
prod_acc = prod_acc.fillna(0)
# Round the accuracies to 2 decimal places
user_acc = np.round(user_acc, 2)
prod_acc = np.round(prod_acc, 2)


px = 1/plt.rcParams['figure.dpi']
plt.rcParams.update({'font.size': 45})

plt.figure(figsize=(5500*px, 2800*px))
grouped_y_clean = grouped_y.replace(0, np.nan)

plt.title('Vegetation Classification Confusion Matrix Level 1\n('+model_run_name+')', fontsize=35, pad=40)
# Add producer's accuracy column to cleaned confusion matrix
grouped_y_clean["Producer's Accuracy"] = prod_acc

# Append NaN value to bottom row of accuracy matrix
user_acc = np.append(user_acc, np.nan)
grouped_y_clean.loc["User's Accuracy"] = user_acc


ax=sn.heatmap(grouped_y_clean, annot=True,cmap='Greens')
ax.set_facecolor('white')
accuracy = np.diag(grouped_y).sum() / grouped_y.to_numpy().sum()
ax.text(0.97, 0.01, str(round(accuracy,2)), transform=ax.transAxes)

sn.set(font_scale=1) 
#plt.title('Vegetation Classification Confusion Matrix Level 3\nUser Accuracy: {}\nProducer Accuracy: {}'.format(user_acc, prod_acc), fontsize=75)
plt.xlabel('Predicted', fontsize=45)
plt.ylabel('Truth', fontsize=45)
plt.yticks(rotation=0, fontsize=45)
plt.xticks(fontsize=45)

pad_val=15


# Apply the customized styling to the class labels
for i, label in enumerate(ax.get_xticklabels()):
    for key in color_obj_level.keys():
        if label.get_text().startswith(key):
            label.set_color(color_obj_level[key]['text'])
            label.set_bbox(dict(facecolor=color_obj_level[key]['backgroud'], edgecolor='none', pad=pad_val))
for i, label in enumerate(ax.get_yticklabels()):
    for key in color_obj_level.keys():
        if label.get_text().startswith(key):
            label.set_color(color_obj_level[key]['text'])
            label.set_bbox(dict(facecolor=color_obj_level[key]['backgroud'], edgecolor='none', pad=pad_val))
    
plt.savefig(os.path.join(output_directory,"cm_count_group_level_1-"+date_str+".png"))

plt.show()

In [ ]:
from sklearn.preprocessing import LabelBinarizer
# Create an instance of the `LabelBinarizer` class and use it to binarize the class labels in `class_names_train`. 
# This is done because feature importance analysis for a multiclass problem requires fitting multiple binary classifiers instead of one multiclass classifier.

lb = LabelBinarizer()
binary_class_names_train = lb.fit_transform(class_names_train)
binary_class_names_train
# Get the number of features and classes 
n_features = variables_train.shape[1]
n_classes = binary_class_names_train.shape[1]

# Initialize an array called `importances_per_class` with zeros for storing feature importances for each class. 
# Its dimensions will be 'number of features' by 'number of classes'.
importances_per_class = np.zeros((n_features, n_classes))

# Loop through each class and fit a `RandomForestClassifier` on the training data, 
# with the binary target variable corresponding to that class. 
# The fitted model is then used to compute the feature importances for that class, 
# which are stored in the relevant column of the `importances_per_class` array
# Loop through each class
for i in range(n_classes):
    # Initialize a new binary RandomForestClassifier model
    binary_model = RandomForestClassifier(class_weight=class_weight)
    
    # Fit the binary model on the training data with the binary target variable corresponding to the current class
    binary_model.fit(variables_train, binary_class_names_train[:, i])

    # Store the feature importances for the current class in the relevant column of the importances_per_class array
    # Populating the empty array with the feature importances for each class
    importances_per_class[:, i] = binary_model.feature_importances_
    


# Initialize an empty list to store unique class labels from the training set
unique_train=[]
# Loop through each element in the class_names_train array, which contains the class labels for the training set
for cls in class_names_train:
    # Convert the class label to an integer, then convert it back to a string.
    # This is done to remove any leading zeros from the label, which may cause issues later
    cls_str = str(int(cls))
    # Check if the string representation of the integer class label is already in the unique_train list
    if cls_str not in unique_train:
            # If it's not, add it to the list
            unique_train.append(cls_str)
# Sort the unique_train list in ascending order
unique_train.sort()

#build a dataframe using the importances_per_class array and the unique_train list

class_imortance_df = pd.DataFrame(importances_per_class, columns =unique_train, dtype = float)
class_imortance_df.insert(0, 'Features', feature_names) 

# set the 'Features' column as the index
class_imortance_df.set_index('Features', inplace=True)

# create the heatmap
fig, ax = plt.subplots(figsize=(40, 20))
sn.heatmap(class_imortance_df, annot=True, cmap='Blues')

# set the x and y axis labels
plt.xlabel('Classes')
plt.ylabel('Features')
plt.title('Class Level Importances\n('+model_run_name+')', fontsize=15, pad=40)
#save the plot
#plt.savefig(os.path.join(output_directory,"Class-Level-Importances-"+date_str+".png"))
plt.savefig(os.path.join(output_directory,"300-dpi-Class-Level-Importances-"+date_str+".png"),dpi=300)

# show the plot
plt.show()

## Classify all segments

In [ ]:
variables_all=segment_df[cols]

variables_all = variables_all.drop('Class',axis='columns')

final_predictions=model.predict(variables_all)

In [ ]:


segment_df['Predicted']=final_predictions

full_testing_df=segment_df.loc[segment_df['training_type'] == 'truth']
full_testing_df

In [ ]:

subset_full_testing_df = full_testing_df[['SegmentID','training_type','PlotID','Class','Predicted']].copy()
subset_full_testing_df.to_csv(os.path.join(output_directory,"test-results-"+date_str+".csv"))

In [ ]:

subset_full_testing_df['match'] = subset_full_testing_df.apply(lambda row: True if row['Class'] == row['Predicted'] else False, axis=1)
subset_full_testing_df.to_csv(os.path.join(output_directory,"test-results-"+date_str+".csv"))

In [ ]:
subset_full_testing_df.to_csv(os.path.join(output_directory,"Full-Predictions-"+date_str+".csv"))

In [ ]:
predicted_df = pd.DataFrame().assign(SegmentID=segment_df['SegmentID'], Predicted=segment_df['Predicted'])
predicted_df=predicted_df.astype(int)
predicted_df.to_csv(os.path.join(output_directory,'predictions-'+date_str+".csv"), index=False)

# Averaged results over many runs (new random 70/30 split each run)

Each run draws a new stratified 70/30 split of the labelled segments (split seed = run number), trains the random forest (random_state = run number, same `class_weight` as above) and scores it on that run's truth segments.
Because the seeds are fixed, re-running gives identical results, and runs with different `class_weight` settings use the same splits and seeds, so they can be compared run for run.

Outputs (prefix `avg<n_runs>-`):
* averaged count confusion matrix (CSV + PNG); margins are user's / producer's accuracy of the averaged matrix
* table of mean ± SD user's and producer's accuracy per class, plus overall accuracy
* averaged level 3, 2 and 1 matrices (CSV + PNG) and their per-group accuracy tables
* per-run overall accuracy and a summary JSON

In [ ]:
n_runs = 100
avg_prefix = f'avg{n_runs}-'


In [ ]:
# All labelled segments, pooled; a new 70/30 split is drawn from these for every run
labelled_df = segment_df.loc[segment_df['training_type'].isin(['training', 'truth'])]
feature_cols = [c for c in cols if c != 'Class']
avg_labels = sorted(int(c) for c in labelled_df['Class'].unique())

run_cms = []
run_scores = []
t0 = time.time()
for run_i in range(n_runs):
    run_split = random_split(labelled_df, train_fraction, seed=run_i)
    run_train = labelled_df[run_split == 'training']
    run_truth = labelled_df[run_split == 'truth']
    y_truth_int = run_truth['Class'].astype(int).to_numpy()

    run_model = RandomForestClassifier(class_weight=class_weight, random_state=run_i, n_jobs=-1)
    run_model.fit(run_train[feature_cols], run_train['Class'])
    run_pred = run_model.predict(run_truth[feature_cols]).astype(int)
    run_cms.append(confusion_matrix(y_truth_int, run_pred, labels=avg_labels))
    run_scores.append(float(np.mean(run_pred == y_truth_int)))
    if (run_i + 1) % 10 == 0:
        print(f'{run_i + 1}/{n_runs} runs done ({time.time() - t0:.0f} s)')
run_cms = np.array(run_cms)          # shape: runs x reference class x classified class
run_scores = np.array(run_scores)
print(f'Overall accuracy: {run_scores.mean():.3f} ± {run_scores.std(ddof=1):.3f} (mean ± SD, {n_runs} runs)')


In [ ]:
def per_run_accuracies(cms):
    """Per-run user's and producer's accuracy for each class. cms: runs x ref x classified.
    Returns NaN where undefined (class absent from truth set / never classified in that run)."""
    diag = np.diagonal(cms, axis1=1, axis2=2).astype(float)
    with np.errstate(divide='ignore', invalid='ignore'):
        producers = diag / cms.sum(axis=2)   # correct / reference (row) total
        users = diag / cms.sum(axis=1)       # correct / classified (column) total
    return users, producers

def fmt_pm(mean, sd):
    return '' if np.isnan(mean) else f'{mean:.2f} ± {sd:.2f}'

def accuracy_table(cms, labels, scores):
    users, producers = per_run_accuracies(cms)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', category=RuntimeWarning)
        ua_mean, ua_sd = np.nanmean(users, axis=0), np.nanstd(users, axis=0, ddof=1)
        pa_mean, pa_sd = np.nanmean(producers, axis=0), np.nanstd(producers, axis=0, ddof=1)
    table = pd.DataFrame({
        'Class': [str(l) for l in labels],
        'Reference segments': cms[0].sum(axis=1),
        "Producer's mean": pa_mean, "Producer's SD": pa_sd,
        "User's mean": ua_mean, "User's SD": ua_sd,
        "Runs class was classified": np.sum(~np.isnan(users), axis=0),
    })
    table["Producer's (mean ± SD)"] = [fmt_pm(m, s) for m, s in zip(pa_mean, pa_sd)]
    table["User's (mean ± SD)"] = [fmt_pm(m, s) for m, s in zip(ua_mean, ua_sd)]
    overall = {'Class': 'Overall accuracy', 'Reference segments': int(cms[0].sum()),
               "Producer's mean": scores.mean(), "Producer's SD": scores.std(ddof=1),
               "User's mean": np.nan, "User's SD": np.nan, 'Runs class was classified': len(scores),
               "Producer's (mean ± SD)": fmt_pm(scores.mean(), scores.std(ddof=1)), "User's (mean ± SD)": ''}
    return pd.concat([table, pd.DataFrame([overall])], ignore_index=True)

def mean_matrix_with_margins(cms, labels):
    mean_cm = cms.mean(axis=0)
    df = pd.DataFrame(mean_cm, index=[str(l) for l in labels], columns=[str(l) for l in labels])
    with np.errstate(divide='ignore', invalid='ignore'):
        pa = np.diag(mean_cm) / mean_cm.sum(axis=1)
        ua = np.diag(mean_cm) / mean_cm.sum(axis=0)
    df["Producer's Accuracy"] = pa
    df.loc["User's Accuracy"] = list(ua) + [np.nan]
    return df

def plot_mean_matrix(df, title, png_path, overall_mean, overall_sd):
    n = len(df.columns)
    plot_df = df.replace(0, np.nan)
    annot = []
    for r, row_vals in enumerate(plot_df.values):
        annot_row = []
        for c, val in enumerate(row_vals):
            is_margin = (r == len(plot_df) - 1) or (c == n - 1)
            annot_row.append('' if np.isnan(val) else (f'{val:.2f}' if is_margin else f'{val:.1f}'))
        annot.append(annot_row)
    size = max(12, n * 1.1)
    fig, ax = plt.subplots(figsize=(size * 1.6, size))
    sn.heatmap(plot_df, annot=np.array(annot), fmt='', cmap='Greens', linewidths=.5, ax=ax,
               annot_kws={'fontsize': 12 if n > 15 else 18})
    ax.set_facecolor('white')
    ax.set_xlabel('Classified', fontsize=24)
    ax.set_ylabel('Reference', fontsize=24)
    ax.tick_params(labelsize=14)
    plt.setp(ax.get_yticklabels(), rotation=0)
    ax.set_title(title + f'\nOverall accuracy {overall_mean:.3f} ± {overall_sd:.3f} (mean ± SD)', fontsize=22, pad=20)
    fig.tight_layout()
    fig.savefig(png_path)
    plt.show()
    plt.close(fig)


In [ ]:
# Finest class level
avg_cm_df = mean_matrix_with_margins(run_cms, avg_labels)
avg_cm_df.to_csv(os.path.join(output_directory, avg_prefix + 'cm_count-' + date_str + '.csv'))
plot_mean_matrix(avg_cm_df,
                 f'Vegetation Classification Confusion Matrix, mean of {n_runs} runs\n({model_run_name})',
                 os.path.join(output_directory, avg_prefix + 'cm_count-' + date_str + '.png'),
                 run_scores.mean(), run_scores.std(ddof=1))

avg_acc_table = accuracy_table(run_cms, avg_labels, run_scores)
avg_acc_table.to_csv(os.path.join(output_directory, avg_prefix + 'user_producer_accuracy-' + date_str + '.csv'), index=False)
avg_acc_table[['Class', 'Reference segments', "User's (mean ± SD)", "Producer's (mean ± SD)", 'Runs class was classified']]


In [ ]:
# Grouped levels: level 3 = first 3 digits of the class code, level 2 = first 2, level 1 = first digit
level_summary = {}
for level in [3, 2, 1]:
    groups = sorted(set(str(l)[:level] for l in avg_labels))
    membership = np.zeros((len(avg_labels), len(groups)))
    for i, l in enumerate(avg_labels):
        membership[i, groups.index(str(l)[:level])] = 1
    # Sum the class-level matrix of every run into groups (rows and columns)
    group_cms = np.einsum('ki,rkl,lj->rij', membership, run_cms, membership)
    group_scores = np.trace(group_cms, axis1=1, axis2=2) / group_cms.sum(axis=(1, 2))
    level_summary[level] = {'mean': float(group_scores.mean()), 'sd': float(group_scores.std(ddof=1))}

    group_df = mean_matrix_with_margins(group_cms, groups)
    group_df.to_csv(os.path.join(output_directory, f'{avg_prefix}cm_count_group_level_{level}-{date_str}.csv'))
    accuracy_table(group_cms, groups, group_scores).to_csv(
        os.path.join(output_directory, f'{avg_prefix}user_producer_accuracy_level_{level}-{date_str}.csv'), index=False)
    plot_mean_matrix(group_df,
                     f'Vegetation Classification Confusion Matrix Level {level}, mean of {n_runs} runs\n({model_run_name})',
                     os.path.join(output_directory, f'{avg_prefix}cm_count_group_level_{level}-{date_str}.png'),
                     group_scores.mean(), group_scores.std(ddof=1))
    print(f'Level {level}: overall accuracy {group_scores.mean():.3f} ± {group_scores.std(ddof=1):.3f}')


In [ ]:
# Per-run scores and a summary of the averaged runs
pd.DataFrame({'run': range(n_runs), 'random_state': range(n_runs), 'split_seed': range(n_runs), 'overall_accuracy': run_scores}).to_csv(
    os.path.join(output_directory, avg_prefix + 'run_scores-' + date_str + '.csv'), index=False)
avg_summary = {
    'run_name': model_run_name,
    'class_weight': class_weight,
    'n_runs': n_runs,
    'random_states': f'0-{n_runs - 1}',
    'split': f'new stratified random split each run, {train_fraction:.0%} of each class for training, split seed = run number',
    'overall_accuracy_mean': float(run_scores.mean()),
    'overall_accuracy_sd': float(run_scores.std(ddof=1)),
    'grouped_levels': {f'level_{k}': v for k, v in level_summary.items()},
    'labelled_segments': int(len(labelled_df)),
    'truth_rows_per_run': int(len(y_truth_int)),
}
with open(os.path.join(output_directory, avg_prefix + 'summary.json'), 'w') as f:
    json.dump(avg_summary, f, indent=4)
avg_summary
